# ecoli2: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('12_ecoli2_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,92.109,87.675,89.364,86.727,97.895,92.311,89.967,96.154,89.892
smote,94.938,89.850,88.030,92.545,97.544,95.045,90.228,96.656,92.394
bsmote,92.812,85.269,81.483,89.727,96.128,92.928,90.570,95.997,89.040
ros,94.519,87.808,83.814,92.545,96.657,94.601,90.145,96.411,91.164
adasyn,94.090,85.957,80.498,92.545,95.777,94.161,90.715,96.548,90.023
casb,94.257,86.798,82.342,92.545,96.134,94.340,89.370,96.181,90.527
casb_relaxed,94.685,88.697,85.748,92.545,97.014,94.780,90.313,96.393,91.691
casb_edited,94.852,89.488,87.389,92.545,97.368,94.957,90.362,96.155,92.170
casb_ros,94.259,86.905,82.605,92.545,96.134,94.340,89.054,95.923,90.582


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,93.135,88.727,89.364,88.727,97.895,93.311,87.682,94.904,90.931
smote,94.258,86.959,82.720,92.545,96.134,94.340,85.807,94.759,90.609
bsmote,91.320,78.808,71.289,90.545,92.271,91.408,82.835,94.054,85.064
ros,93.635,84.620,79.357,92.545,94.909,93.727,86.409,94.850,89.128
adasyn,91.232,75.373,64.609,92.545,90.163,91.354,85.501,94.500,83.302
casb,93.554,83.804,77.359,92.545,94.731,93.638,88.736,95.055,88.679
casb_relaxed,93.992,85.629,80.339,92.545,95.608,94.077,86.604,94.833,89.811
casb_edited,93.725,84.368,78.183,92.545,95.081,93.813,85.132,94.672,89.046
casb_ros,93.552,83.875,77.596,92.545,94.731,93.638,86.694,94.888,88.713


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,85.075,81.803,93.056,73.273,98.947,86.110,88.631,95.920,83.439
smote,88.441,83.774,89.170,80.273,98.070,89.171,90.425,96.746,86.107
bsmote,87.631,79.610,80.732,80.364,96.310,88.337,88.923,95.442,83.620
ros,88.336,82.976,87.383,80.273,97.716,88.994,90.122,96.324,85.656
adasyn,88.123,80.206,80.823,81.182,96.310,88.746,89.825,95.980,84.164
casb,90.497,85.717,88.864,84.091,97.895,90.993,91.455,96.190,88.107
casb_relaxed,90.444,85.467,88.426,84.000,97.719,90.860,91.630,96.872,87.955
casb_edited,90.265,84.572,86.342,84.182,97.368,90.775,91.517,96.518,87.418
casb_ros,90.873,85.564,87.537,84.909,97.544,91.226,91.015,96.296,88.219


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,76.871,68.455,77.619,61.455,96.836,79.145,81.387,93.438,72.663
smote,89.157,74.391,65.137,87.545,91.024,89.285,81.342,93.509,81.774
bsmote,87.122,68.603,56.980,86.727,87.682,87.204,79.464,92.814,77.862
ros,88.894,73.388,63.601,87.545,90.323,88.934,81.225,93.497,81.141
adasyn,89.733,72.784,61.384,90.455,89.095,89.775,81.681,93.490,81.259
casb,89.925,74.882,65.206,89.455,90.495,89.975,80.504,93.296,82.403
casb_relaxed,89.061,73.827,64.224,87.545,90.674,89.110,81.315,93.452,81.444
casb_edited,90.364,74.034,62.751,91.364,89.439,90.401,81.068,93.452,82.199
casb_ros,89.453,74.382,64.986,88.545,90.495,89.520,80.840,93.349,81.918


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,86.797,81.665,87.350,77.545,97.893,87.719,86.917,95.104,84.231
smote,91.699,83.744,81.264,88.227,95.693,91.960,86.951,95.418,87.721
bsmote,89.721,78.072,72.621,86.841,93.098,89.969,85.448,94.577,83.897
ros,91.346,82.198,78.539,88.227,94.901,91.564,86.975,95.271,86.772
adasyn,90.795,78.580,71.829,89.182,92.836,91.009,86.931,95.130,84.687
casb,92.058,82.800,78.443,89.659,94.814,92.236,87.516,95.180,87.429
casb_relaxed,92.046,83.405,79.684,89.159,95.254,92.206,87.465,95.387,87.725
casb_edited,92.301,83.115,78.666,90.159,94.814,92.487,87.020,95.199,87.708
casb_ros,92.034,82.681,78.181,89.636,94.726,92.181,86.901,95.114,87.358


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,94.852,smote,94.938,-0.086,LOSS
1,f1,casb_edited,89.488,smote,89.850,-0.362,LOSS
2,precision,casb_edited,87.389,smote,88.030,-0.641,LOSS
3,recall,casb,92.545,smote,92.545,0.000,TIE


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_relaxed,93.992,smote,94.258,-0.266,LOSS
1,f1,casb_relaxed,85.629,smote,86.959,-1.330,LOSS
2,precision,casb_relaxed,80.339,smote,82.720,-2.381,LOSS
3,recall,casb,92.545,smote,92.545,0.000,TIE


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_ros,90.873,smote,88.441,2.432,WIN
1,f1,casb,85.717,smote,83.774,1.943,WIN
2,precision,casb,88.864,smote,89.170,-0.306,LOSS
3,recall,casb_ros,84.909,adasyn,81.182,3.727,WIN


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,92.301,smote,91.699,0.603,WIN
1,f1,casb_relaxed,83.405,smote,83.744,-0.339,LOSS
2,precision,casb_relaxed,79.684,smote,81.264,-1.580,LOSS
3,recall,casb_edited,90.159,adasyn,89.182,0.977,WIN


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.888523,0.818182,0.818182,0.818182,0.964912,0.891547,0.882992,0.928230,0.853353,None
1,42,1,svm,smote,0.50,0.936586,0.869565,0.833333,0.909091,0.964912,0.937002,0.906588,0.939394,0.903076,None
2,42,1,svm,bsmote,0.50,0.880409,0.782609,0.750000,0.818182,0.947368,0.882775,0.874344,0.917065,0.831509,None
3,42,1,svm,ros,0.50,0.936586,0.869565,0.833333,0.909091,0.964912,0.937002,0.915545,0.947368,0.903076,None
4,42,1,svm,adasyn,0.50,0.919399,0.800000,0.714286,0.909091,0.929825,0.919458,0.915679,0.940989,0.859700,None
5,42,1,svm,casb,0.50,0.928032,0.833333,0.769231,0.909091,0.947368,0.928230,0.907071,0.940989,0.880683,None
6,42,1,svm,casb_relaxed,0.50,0.928032,0.833333,0.769231,0.909091,0.947368,0.928230,0.914463,0.944179,0.880683,None
7,42,1,svm,casb_edited,0.50,0.936586,0.869565,0.833333,0.909091,0.964912,0.937002,0.907071,0.940989,0.903076,None
8,42,1,svm,casb_ros,0.50,0.928032,0.833333,0.769231,0.909091,0.947368,0.928230,0.903716,0.928230,0.880683,None
9,42,1,knn,original,0.50,0.888523,0.818182,0.818182,0.818182,0.964912,0.891547,0.865103,0.936204,0.853353,None
